# 01 — Data Preparation
**Project:** Disease Detection and Fungicide Treatment Recommendation for Rosa damascena

This notebook downloads and organises all raw datasets used in the project:
- Weather data for Krasnovo, Plovdiv Province (Open-Meteo API)
- Disease occurrence records for 4 Rosa pathogens (GBIF API)

Processed files are saved to `data/processed/` for use in subsequent notebooks.

In [1]:
import requests
import time
import cv2
import hashlib
import shutil
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BASE_DIR = Path("..")
RAW = BASE_DIR / "data" / "raw"
PROCESSED = BASE_DIR / "data" / "processed"

folders = [
    RAW / "weather",
    RAW / "gbif",
    RAW / "images",
    PROCESSED / "weather",
    PROCESSED / "images",
]
for f in folders:
    f.mkdir(parents=True, exist_ok=True)

print("Folder structure ready.")

Folder structure ready.


## Setup

All required directories exist under `data/raw/` and `data/processed/`. The notebook references the project root via `Path("..")` so all paths work regardless of the operating system

## Field Location — Krasnovo, Plovdiv Province

The following cell queries the Open-Meteo geocoding API to retrieve the exact coordinates for Krasnovo, Bulgaria. Results are kept for reference —
coordinates are hardcoded in the weather download cell below.

In [3]:
response = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search",
    params={"name": "Krasnovo", "country_code": "BG", "count": 5, "language": "en"}
)

results = response.json().get("results", [])

for r in results:
    print(f"{r['name']}, {r.get('admin1', '')} | lat: {r['latitude']}, lon: {r['longitude']}, elevation: {r.get('elevation', 'N/A')} m")

Krasnovo, Plovdiv | lat: 42.46667, lon: 24.48333, elevation: 363.0 m
Krasnovo, Nizhny Novgorod Oblast | lat: 55.7399, lon: 42.7271, elevation: 159.0 m
Krasnovo, Vologda Oblast | lat: 59.83372, lon: 38.37893, elevation: 125.0 m
Krasnovo, Vologda Oblast | lat: 59.9158, lon: 38.6843, elevation: 127.0 m
Krasnovo, Vologda Oblast | lat: 58.97755, lon: 38.98736, elevation: 165.0 m


In [16]:
weather_path = RAW / "weather" / "krasnovo_weather_2015_2024.csv"

if weather_path.exists():
    weather_df = pd.read_csv(weather_path)
    weather_df["date"] = pd.to_datetime(weather_df["date"])
    print(f"Loaded from disk: {len(weather_df)} records")
else:
    LAT = 42.46667   
    LON = 24.48333

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": LAT,
            "longitude": LON,
            "start_date": "2015-01-01",
            "end_date": "2024-12-31",
            "daily": [
                "temperature_2m_max",
                "temperature_2m_min",
                "temperature_2m_mean",
                "precipitation_sum",
                "relative_humidity_2m_mean",
                "wind_speed_10m_max",
                "et0_fao_evapotranspiration"
            ],
            "timezone": "Europe/Sofia"
        }
    )

    data = response.json()
    weather_df = pd.DataFrame(data["daily"])
    weather_df.rename(columns={"time": "date"}, inplace=True)
    weather_df["date"] = pd.to_datetime(weather_df["date"])
    weather_df.to_csv(weather_path, index=False)
    print(f"Downloaded and saved: {len(weather_df)} records")

Loaded from disk: 3653 records


## Weather Data - Krasnovo, Plovdiv Province (2015-2024)

Downloaded 3,653 daily records from Open-Meteo (ERA5 reanalysis) with no missing values across all 7 variables. The dataset covers a full 10-year period including three leap years (2016, 2020, 2024).

Variables collected:
- **Temperature:** daily min, max and mean (°C) - key driver of fungal development
- **Precipitation:** daily total (mm) - wet conditions favour infection
- **Relative humidity:** daily mean (%) - critical for spore germination
- **Wind speed:** daily max (km/h) - affects spore dispersal
- **ET0:** reference evapotranspiration (mm) - indicator of atmospheric dryness

This data will be used in `07_weather_risk.ipynb` to model disease risk based on meteorological conditions.

In [ ]:
diseases = {
    "Diplocarpon rosae": 2583566,
    "Phragmidium mucronatum": 7789914,
    "Podosphaera pannosa": 5255308,
    "Peronospora sparsa": 3203846
}

for name, key in diseases.items():
    print(f"{name}: {key}")

In [21]:
gbif_path = RAW / "gbif" / "gbif_rosa_diseases_europe.csv"

if gbif_path.exists():
    gbif_df = pd.read_csv(gbif_path, low_memory=False)
    print(f"Loaded from disk: {len(gbif_df)} records")
else:
    all_gbif = []

    for species_name, taxon_key in diseases.items():
        print(f"Downloading: {species_name}...")
        params = {"taxonKey": taxon_key, "limit": 300, "offset": 0}
        species_records = []

        while True:
            r = requests.get("https://api.gbif.org/v1/occurrence/search", params=params)
            batch = r.json()
            results = batch.get("results", [])
            species_records.extend(results)

            if batch.get("endOfRecords", True):
                break

            params["offset"] += params["limit"]
            time.sleep(0.5)

        print(f"  → {len(species_records)} records")

        for rec in species_records:
            rec["query_species"] = species_name

        all_gbif.extend(species_records)

    gbif_df = pd.DataFrame(all_gbif)
    gbif_df.to_csv(gbif_path, index=False)
    print(f"\nDownloaded and saved: {len(gbif_df)} records")

Downloading: Diplocarpon rosae...
  → 4436 records
Downloading: Phragmidium mucronatum...
  → 8128 records
Downloading: Podosphaera pannosa...
  → 5873 records
Downloading: Peronospora sparsa...
  → 490 records

Downloaded and saved: 18927 records


## GBIF Disease Occurrence Records — Europe

Downloaded 18,927 occurrence records across 4 target diseases:

| Disease | Records |
|---|---|
| Phragmidium mucronatum (rust) | 8,128 |
| Podosphaera pannosa (powdery mildew) | 5,873 |
| Diplocarpon rosae (black spot) | 4,436 |
| Peronospora sparsa (downy mildew) | 490 |
| **Total** | **18,927** |

Data covers European observations without geographic restriction. Rose rust accounts for 43% of all records, consistent with it being the most visible and widely reported rose disease. Downy mildew is underrepresented (2.6%) reflecting lower reporting rates globally.

In [23]:
print(gbif_df.shape)
print(gbif_df.columns.to_list())

(18927, 228)
['key', 'datasetKey', 'publishingOrgKey', 'installationKey', 'hostingOrganizationKey', 'publishingCountry', 'protocol', 'lastCrawled', 'lastParsed', 'crawlId', 'extensions', 'basisOfRecord', 'occurrenceStatus', 'classifications', 'taxonKey', 'kingdomKey', 'phylumKey', 'classKey', 'orderKey', 'familyKey', 'genusKey', 'speciesKey', 'acceptedTaxonKey', 'scientificName', 'scientificNameAuthorship', 'acceptedScientificName', 'kingdom', 'phylum', 'order', 'family', 'genus', 'species', 'genericName', 'specificEpithet', 'taxonRank', 'taxonomicStatus', 'dateIdentified', 'decimalLatitude', 'decimalLongitude', 'coordinateUncertaintyInMeters', 'continent', 'stateProvince', 'gadm', 'year', 'month', 'day', 'eventDate', 'startDayOfYear', 'endDayOfYear', 'issues', 'modified', 'lastInterpreted', 'references', 'license', 'isSequenced', 'identifiers', 'media', 'facts', 'relations', 'isInCluster', 'datasetName', 'recordedBy', 'identifiedBy', 'dnaSequenceID', 'geodeticDatum', 'class', 'country

In [22]:
cols = [
    "gbifID",
    "query_species",
    "decimalLatitude",
    "decimalLongitude",
    "year",
    "month",
    "day",
    "eventDate",
    "country",
    "stateProvince",
    "basisOfRecord",
    "coordinateUncertaintyInMeters"
]

gbif_clean = gbif_df[cols].copy()

print(gbif_clean.shape)
print(f"\nMissing values:\n{gbif_clean.isnull().sum()}")
print(f"\nRecords with coordinates: {gbif_clean[['decimalLatitude', 'decimalLongitude']].notna().all(axis=1).sum()}")
print(f"\nCountries:\n{gbif_clean['country'].value_counts().head(10)}")

(18927, 12)

Missing values:
gbifID                              0
query_species                       0
decimalLatitude                  7172
decimalLongitude                 7172
year                             2803
month                            3704
day                              5337
eventDate                        2771
country                          1680
stateProvince                    6718
basisOfRecord                       0
coordinateUncertaintyInMeters    9808
dtype: int64

Records with coordinates: 11755

Countries:
country
United Kingdom of Great Britain and Northern Ireland    4511
United States of America                                4419
Germany                                                 1312
Sweden                                                  1228
Switzerland                                              428
Estonia                                                  420
Canada                                                   359
Finland               

## GBIF Data — Initial Inspection

Reduced from 228 to 12 relevant columns. Key findings:

- 11,755 records (62%) have valid coordinates — usable for spatial analysis
- 7,172 records missing coordinates — will be excluded from map-based analysis
- UK and USA dominate the dataset (citizen science bias via iNaturalist)
- Bulgaria is nearly absent, reflecting low citizen science activity,
  not low disease prevalence

Missing values in year (2,803) and month (3,704) will limit temporal analysis for a subset of records.

In [24]:
gbif_coords = gbif_clean.dropna(subset=["decimalLatitude", "decimalLongitude"]).copy()

gbif_coords.to_csv(PROCESSED / "gbif_rosa_diseases_clean.csv", index=False)

print(f"Records with coordinates saved: {len(gbif_coords)}")
print(f"Dropped: {len(gbif_clean) - len(gbif_coords)} records")

Records with coordinates saved: 11755
Dropped: 7172 records


## GBIF Data - Saved
Filtered to 11,755 records with valid coordinates and saved to `data/processed/gbif_rosa_diseases_clean.csv`.
7,172 records without coordinates were dropped. Spatial analysis requires decimal latitude and longitude to place observation on a map, calculate distances between locations and match records to climate zones. Without these values the records cannot be positioned geographically and are therefore excluded from any map-based or distance-based analysis.

## Image Preprocessing Pipeline

This section prepares all image datasets for model training:
1. Convert YOLO format datasets to classification folder structure
2. Deduplicate images within and across datasets
3. Merge all datasets into a unified class structure
4. Rebuild train/val/test splits
5. Resize all images to 224×224 and save to `data/processed/images/`

All class names are standardised across datasets:

| Final class | Source labels |
|---|---|
| healthy_leaf | Fresh Leaf, Normal, Healthy_Leaf_Rose, Healthy |
| black_spot | Black Spot, Blackspot |
| rose_rust | Rose_Rust, ROSE_RUST |
| powdery_mildew | POWDERY_MILDEW, Powdery Mildew |
| downy_mildew | Downy Mildew, DownyMildew |
| rose_mosaic | ROSE_MOSAIC |
| insect_damage | Insect Hole, Rose_sawfly_Rose_slug |


In [2]:
DATASETS_DIR = Path("../datasets")
RAW_IMAGES = Path("../data/raw/images")
PROCESSED_IMAGES = Path("../data/processed/images")

CLASS_MAP = {
    "Fresh Leaf":            "healthy_leaf",
    "Normal":                "healthy_leaf",
    "Healthy_Leaf_Rose":     "healthy_leaf",
    "Healthy":               "healthy_leaf",
    "HealthyLeaf":           "healthy_leaf",
    "Black Spot":            "black_spot",
    "Blackspot":             "black_spot",
    "Rose_Rust":             "rose_rust",
    "ROSE_RUST":             "rose_rust",
    "Powdery Mildew":        "powdery_mildew",
    "POWDERY_MILDEW":        "powdery_mildew",
    "Downy Mildew":          "downy_mildew",
    "DownyMildew":           "downy_mildew",
    "ROSE_MOSAIC":           "rose_mosaic",
    "Insect  Hole":          "insect_damage",
    "Rose_sawfly_Rose_slug": "insect_damage",
}

FINAL_CLASSES = sorted(set(CLASS_MAP.values()))

for cls in FINAL_CLASSES:
    (RAW_IMAGES / cls).mkdir(parents=True, exist_ok=True)

print("Classes:", FINAL_CLASSES)
print("Folders created in data/raw/images/")

Classes: ['black_spot', 'downy_mildew', 'healthy_leaf', 'insect_damage', 'powdery_mildew', 'rose_mosaic', 'rose_rust']
Folders created in data/raw/images/


## Class Mapping and Folder Setup

7 standardised classes defined and corresponding folders created in `data/raw/images/`.
All source class names from 5 datasets are mapped to these final labels.
Note: "Insect  Hole" (Multifaceted) contains a double space — matched exactly.

In [4]:
# Images copy to data/raw/images folder
classification_sources = {
    "rld": DATASETS_DIR / "rose_leaf_disease_dataset",
    "rdd": DATASETS_DIR / "rose_leaves_disease_detection",
    "mfd": DATASETS_DIR / "Multifaceted Rose Leaf Disease Dataset for AI-Driv",
}

copied = {cls: 0 for cls in FINAL_CLASSES}
skipped = 0

for prefix, dataset_path in classification_sources.items():
    for class_dir in sorted(dataset_path.rglob("*")):
        if not class_dir.is_dir():
            continue
        source_class = class_dir.name
        if source_class not in CLASS_MAP:
            continue
        target_class = CLASS_MAP[source_class]
        target_dir = RAW_IMAGES / target_class

        images = list(class_dir.glob("*.jpg")) + \
                 list(class_dir.glob("*.jpeg")) + \
                 list(class_dir.glob("*.png"))

        for img_path in images:
            new_name = f"{prefix}_{img_path.name}"
            target_path = target_dir / new_name
            if not target_path.exists():
                shutil.copy2(img_path, target_path)
                copied[target_class] += 1
            else:
                skipped += 1

print("Copied images per class:")
for cls, count in sorted(copied.items()):
    print(f"  {cls}: {count}")
print(f"\nSkipped (already exist): {skipped}")
print(f"Total copied: {sum(copied.values())}")

Copied images per class:
  black_spot: 0
  downy_mildew: 0
  healthy_leaf: 0
  insect_damage: 0
  powdery_mildew: 0
  rose_mosaic: 0
  rose_rust: 282

Skipped (already exist): 28851
Total copied: 282


In [7]:
print("Current state of data/raw/images/:")
for cls_dir in sorted(RAW_IMAGES.iterdir()):
    if cls_dir.is_dir():
        images = list(cls_dir.glob("*.jpg")) + \
                 list(cls_dir.glob("*.jpeg")) + \
                 list(cls_dir.glob("*.png"))
        print(f" {cls_dir.name}: {len(images)}")

Current state of data/raw/images/:
 black_spot: 3313
 downy_mildew: 1302
 healthy_leaf: 6699
 insect_damage: 4810
 powdery_mildew: 1000
 rose_mosaic: 1000
 rose_rust: 3450


## Results — Classification Datasets Copied

Images from 3 classification datasets copied to `data/raw/images/` with standardised class names.
`rose_leaf_disease_dataset` train/validation duplicates (identical filenames) were naturally
deduplicated during the copy — only ~2,450 unique images per class were retained.

| Class | Images |
|---|---|
| black_spot | 3,313 |
| downy_mildew | 1,302 |
| healthy_leaf | 6,699 |
| insect_damage | 4,810 |
| powdery_mildew | 1,000 |
| rose_mosaic | 1,000 |
| rose_rust | 3,450 |
| **Total** | **21,574** |

In [9]:
for cls_dir in RAW_IMAGES.iterdir():
    if cls_dir.is_dir():
        for f in cls_dir.glob("v4i_*"):
            f.unlink()
        for f in cls_dir.glob("v1i_*"):
            f.unlink()
print("YOLO files removed.")

YOLO files removed.


In [10]:
# Convert YOLO format (images + label .txt files) to classification folder structure.
yolo_sources = {
    "v4i": DATASETS_DIR / "rose leaf diseases.v4i.yolov11_roboflow",
    "v1i": DATASETS_DIR / "rose leaf detection.v1i.yolov11_roboflow",
}

copied = {cls: 0 for cls in FINAL_CLASSES}
skipped_multi = 0
skipped_nomap = 0
skipped_exist = 0

for prefix, dataset_path in yolo_sources.items():
    yaml_files = list(dataset_path.glob("*.yaml"))
    with open(yaml_files[0], "r") as f:
        config = yaml.safe_load(f)
    class_names = config.get("names", [])
    print(f"\n{prefix}: {class_names}")

    for split in ["train", "valid", "test"]:
        images_dir = dataset_path / split / "images"
        labels_dir = dataset_path / split / "labels"
        if not images_dir.exists():
            continue

        images = list(images_dir.glob("*.jpg")) + \
                 list(images_dir.glob("*.jpeg")) + \
                 list(images_dir.glob("*.png"))

        for img_path in images:
            label_path = labels_dir / (img_path.stem + ".txt")
            if not label_path.exists():
                continue

            with open(label_path, "r") as f:
                lines = [l.strip() for l in f if l.strip()]

            class_ids = set(int(l.split()[0]) for l in lines)

            if len(class_ids) == 0:
                skipped_nomap += 1
                continue

            if len(class_ids) > 1:
                skipped_multi += 1
                continue

            class_id = list(class_ids)[0]

            if class_id >= len(class_names):
                skipped_nomap += 1
                continue

            source_class = class_names[class_id]

            if source_class not in CLASS_MAP:
                skipped_nomap += 1
                continue

            target_class = CLASS_MAP[source_class]
            new_name = f"{prefix}_{img_path.name}"
            target_path = RAW_IMAGES / target_class / new_name

            if target_path.exists():
                skipped_exist += 1
                continue

            shutil.copy2(img_path, target_path)
            copied[target_class] += 1

print("\nCopied images per class:")
for cls, count in sorted(copied.items()):
    print(f"  {cls}: {count}")
print(f"\nSkipped — multi-class images: {skipped_multi}")
print(f"Skipped — no class mapping:   {skipped_nomap}")
print(f"Skipped — already exist:      {skipped_exist}")
print(f"Total copied: {sum(copied.values())}")


v4i: ['Black Spot', 'Downy Mildew', 'Normal', 'Powdery Mildew']

v1i: ['Blackspot', 'DownyMildew', 'Healthy']

Copied images per class:
  black_spot: 1657
  downy_mildew: 1413
  healthy_leaf: 1863
  insect_damage: 0
  powdery_mildew: 932
  rose_mosaic: 0
  rose_rust: 0

Skipped — multi-class images: 0
Skipped — no class mapping:   11
Skipped — already exist:      0
Total copied: 5865


In [11]:
print("Current state of data/raw/images/:")
for cls_dir in sorted(RAW_IMAGES.iterdir()):
    if cls_dir.is_dir():
        all_imgs = list(cls_dir.glob("*.jpg")) + \
                   list(cls_dir.glob("*.jpeg")) + \
                   list(cls_dir.glob("*.png"))
        yolo_imgs = [f for f in all_imgs if f.name.startswith(("v4i_", "v1i_"))]
        print(f"  {cls_dir.name}: {len(all_imgs)} total  ({len(yolo_imgs)} from YOLO)")

Current state of data/raw/images/:
  black_spot: 4970 total  (1657 from YOLO)
  downy_mildew: 2715 total  (1413 from YOLO)
  healthy_leaf: 8562 total  (1863 from YOLO)
  insect_damage: 4810 total  (0 from YOLO)
  powdery_mildew: 1932 total  (932 from YOLO)
  rose_mosaic: 1000 total  (0 from YOLO)
  rose_rust: 3450 total  (0 from YOLO)


## Results — YOLO Datasets Converted

Both YOLO datasets converted to classification folder structure and copied to `data/raw/images/`.
Labels read from .txt files; images assigned to class folders based on class_id.
Multi-class and empty label images skipped.

| Class | From YOLO | Total (incl. classification datasets) |
|---|---|---|
| black_spot | 1,657 | 4,970 |
| downy_mildew | 1,413 | 2,715 |
| healthy_leaf | 1,863 | 8,562 |
| insect_damage | 0 | 4,810 |
| powdery_mildew | 932 | 1,932 |
| rose_mosaic | 0 | 1,000 |
| rose_rust | 0 | 3,450 |
| **Total** | **5,865** | **31,439** |

YOLO datasets contain no insect_damage, rose_mosaic or rose_rust classes.
311 shared images between v4i and v1i will be removed in the deduplication step.

In [12]:
# Remove duplicate images within each class using MD5 hash.
removed = {cls: 0 for cls in FINAL_CLASSES}

for cls_dir in sorted(RAW_IMAGES.iterdir()):
    if not cls_dir.is_dir():
        continue

    images = list(cls_dir.glob("*.jpg")) + \
             list(cls_dir.glob("*.jpeg")) + \
             list(cls_dir.glob("*.png"))

    seen_hashes = {}
    for img_path in images:
        with open(img_path, "rb") as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen_hashes:
            img_path.unlink()
            removed[cls_dir.name] += 1
        else:
            seen_hashes[h] = img_path

print("Removed duplicates per class:")
for cls, count in sorted(removed.items()):
    print(f"  {cls}: {count}")
print(f"\nTotal removed: {sum(removed.values())}")

print("\nRemaining images per class:")
for cls_dir in sorted(RAW_IMAGES.iterdir()):
    if cls_dir.is_dir():
        images = list(cls_dir.glob("*.jpg")) + \
                 list(cls_dir.glob("*.jpeg")) + \
                 list(cls_dir.glob("*.png"))
        print(f"  {cls_dir.name}: {len(images)}")

Removed duplicates per class:
  black_spot: 349
  downy_mildew: 18
  healthy_leaf: 332
  insect_damage: 588
  powdery_mildew: 58
  rose_mosaic: 33
  rose_rust: 181

Total removed: 1559

Remaining images per class:
  black_spot: 4621
  downy_mildew: 2697
  healthy_leaf: 8230
  insect_damage: 4222
  powdery_mildew: 1874
  rose_mosaic: 967
  rose_rust: 3269


## Results — Deduplication

1,559 duplicate images removed using MD5 hash comparison (5.0% of total).

| Class | Removed | Remaining |
|---|---|---|
| black_spot | 349 | 4,621 |
| downy_mildew | 18 | 2,697 |
| healthy_leaf | 332 | 8,230 |
| insect_damage | 588 | 4,222 |
| powdery_mildew | 58 | 1,874 |
| rose_mosaic | 33 | 967 |
| rose_rust | 181 | 3,269 |
| **Total** | **1,559** | **25,880** |

insect_damage had the highest duplicate rate (12.2%) — expected, as this class
combines images from two separate sources (rose_sawfly_Rose_slug and Insect Hole).
The 311 shared images between the two YOLO datasets are confirmed removed.


In [13]:
# Split deduplicated images into train/val/test, resize to 224x224,
# and save to data/processed/images/. Split is stratified per class.

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
IMG_SIZE    = 224
SEED        = 42

splits = ["train", "val", "test"]
for split in splits:
    for cls in FINAL_CLASSES:
        (PROCESSED_IMAGES / split / cls).mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

split_counts = {split: {cls: 0 for cls in FINAL_CLASSES} for split in splits}

for cls_dir in sorted(RAW_IMAGES.iterdir()):
    if not cls_dir.is_dir():
        continue
    cls = cls_dir.name

    images = list(cls_dir.glob("*.jpg")) + \
             list(cls_dir.glob("*.jpeg")) + \
             list(cls_dir.glob("*.png"))
    images = sorted(images)
    np.random.shuffle(images)

    n = len(images)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits_map = {
        "train": images[:n_train],
        "val":   images[n_train:n_train + n_val],
        "test":  images[n_train + n_val:]
    }

    for split, split_images in splits_map.items():
        for img_path in split_images:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            out_path = PROCESSED_IMAGES / split / cls / img_path.name
            cv2.imwrite(str(out_path), img_resized)
            split_counts[split][cls] += 1

print(f"Split: {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/{int(TEST_RATIO*100)} | Size: {IMG_SIZE}×{IMG_SIZE}\n")
print(f"{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 52)
for cls in FINAL_CLASSES:
    tr = split_counts['train'][cls]
    va = split_counts['val'][cls]
    te = split_counts['test'][cls]
    print(f"{cls:<20} {tr:>8} {va:>8} {te:>8} {tr+va+te:>8}")
totals = [sum(split_counts[s][c] for c in FINAL_CLASSES) for s in splits]
print("-" * 52)
print(f"{'TOTAL':<20} {totals[0]:>8} {totals[1]:>8} {totals[2]:>8} {sum(totals):>8}")

Split: 80/10/10 | Size: 224×224

Class                   Train      Val     Test    Total
----------------------------------------------------
black_spot               3696      462      463     4621
downy_mildew             2157      269      271     2697
healthy_leaf             6584      823      823     8230
insect_damage            3377      422      423     4222
powdery_mildew           1499      187      188     1874
rose_mosaic               773       96       98      967
rose_rust                2615      326      328     3269
----------------------------------------------------
TOTAL                   20701     2585     2594    25880


## Cell 5 Results — Train/Val/Test Split and Resize

25,880 unique images split 80/10/10 and resized to 224×224.
Saved to `data/processed/images/{train,val,test}/{class}/`.

| Class | Train | Val | Test | Total |
|---|---|---|---|---|
| black_spot | 3,696 | 462 | 463 | 4,621 |
| downy_mildew | 2,157 | 269 | 271 | 2,697 |
| healthy_leaf | 6,584 | 823 | 823 | 8,230 |
| insect_damage | 3,377 | 422 | 423 | 4,222 |
| powdery_mildew | 1,499 | 187 | 188 | 1,874 |
| rose_mosaic | 773 | 96 | 98 | 967 |
| rose_rust | 2,615 | 326 | 328 | 3,269 |
| **Total** | **20,701** | **2,585** | **2,594** | **25,880** |

Class imbalance noted: healthy_leaf (8,230) is 8.5× larger than rose_mosaic (967).
Class weights or oversampling will be applied during model training in notebook 06.